1. Import Libraries

In [9]:
import numpy as np
import pandas as pd
import re

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

Cell 2: Load Data

In [10]:
df = pd.read_csv("/kaggle/input/datasets/ggtejas/tmdb-imdb-merged-movies-dataset/TMDB  IMDB Movies Dataset.csv")
df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2812053,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2522038,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3162344,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ...",James Cameron,James Cameron,7.9,1495917,"Sam Worthington, Zoe Saldaña, Sigourney Weaver..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com...",Joss Whedon,"Joss Whedon, Zak Penn",8.0,1556403,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ..."


Cell 3: Create Text Column

In [11]:
df['text'] = df['genres'] + " " + df['keywords'] + " " + df['cast']
df = df[['text', 'vote_average']]
df.dropna(inplace=True)

Cell 4: Clean Text

In [12]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['text'] = df['text'].apply(clean_text)

Cell 5: Create Labels

In [13]:
df['label'] = df['vote_average'].apply(lambda x: 1 if x >= 7 else 0)

Cell 6: Train-Test Split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

Cell 7: Tokenization

In [15]:
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

Cell 8: Padding

In [16]:
max_len = 150

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

Cell 9: SimpleRNN Model

In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

model = Sequential()

model.add(Embedding(input_dim=10000, output_dim=64))

model.add(SimpleRNN(64, activation='tanh'))

model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


model.build(input_shape=(None, 150))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 150, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 648,321 (2.47 MB)

 Trainable params: 648,321 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

Cell 10: Train Model

In [18]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
2968/2968 ━━━━━━━━━━━━━━━━━━━━ 126s 42ms/step - accuracy: 0.8037 - loss: 0.4968 - val_accuracy: 0.8058 - val_loss: 0.4933
Epoch 2/5
2968/2968 ━━━━━━━━━━━━━━━━━━━━ 116s 39ms/step - accuracy: 0.8063 - loss: 0.4926 - val_accuracy: 0.8058 - val_loss: 0.4921
Epoch 3/5
2968/2968 ━━━━━━━━━━━━━━━━━━━━ 115s 39ms/step - accuracy: 0.8066 - loss: 0.4922 - val_accuracy: 0.8058 - val_loss: 0.4927
Epoch 4/5
2968/2968 ━━━━━━━━━━━━━━━━━━━━ 116s 39ms/step - accuracy: 0.8061 - loss: 0.4929 - val_accuracy: 0.8058 - val_loss: 0.4943
Epoch 5/5
2968/2968 ━━━━━━━━━━━━━━━━━━━━ 115s 39ms/step - accuracy: 0.8080 - loss: 0.4899 - val_accuracy: 0.8058 - val_loss: 0.4961


Cell 11: Evaluate

In [19]:
loss, acc = model.evaluate(X_test_pad, y_test)
print("Accuracy:", acc)

928/928 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - accuracy: 0.8149 - loss: 0.4853
Accuracy: 0.812131404876709


Cell 12: Prediction

In [20]:
def predict_movie(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')
    
    pred = model.predict(pad)[0][0]
    
    return "Good Movie" if pred > 0.5 else "Bad Movie"

Test

In [21]:
predict_movie("action superhero marvel avengers ironman")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step


'Bad Movie'